# 🎯 [K12_M01] 자주 묻는 행정 질문 파악하기
### 학생 실습 워크북 & 프롬프트 설계 가이드 (Google Colab / Jupyter)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K12/k12_m01_student.ipynb)

---

## 📌 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 / 과제** | 일반행정·부대관리 (Level 1) / K12 행정 업무 FAQ 챗봇 설계 |
| **미션** | 미션 01. 자주 묻는 행정 질문 파악하기 (소요시간: 약 1.5시간) |
| **실습 유형** | 생성형 AI 시스템 프롬프트 엔지니어링 및 팩트체크 검증 워크북 |

> **⚠️ [교육용 가상 자료 원칙]**
> 본 실습의 모든 자료는 안전 규정에 따라 **교육용 가상 부대 자료(가상 한빛대대)**로 구성되어 있습니다.
> 실제 군사 기밀이나 부대 세부 인적·작전 정보는 외부 AI 도구에 절대 입력하지 않습니다.



In [ ]:
# ============================================================================
# [환경 설정 및 헬퍼 함수] 마크다운 시각화 렌더러 및 가상 데이터셋 로더
# ============================================================================
from IPython.display import display, Markdown
import json
import os

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드하여 반환합니다."""
    filename = f"k12_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K12", "../uploads/K12"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_dataset_summary(mission_num):
    """해당 미션의 데이터셋 주요 항목을 요약 출력합니다."""
    data = load_dataset(mission_num)
    if data:
        display(Markdown(f"**[데이터셋 로드 성공]** `{data.get('title', '')}` ({data.get('mission', '')})"))
        keys = [k for k in data.keys() if k not in ["project", "mission", "title"]]
        print(f"포함 항목: {', '.join(keys)}")

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 AI 답변 및 검증 결과를 시각적으로 렌더링하는 헬퍼 함수"""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 AI 생성 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ 워크북 시각화 헬퍼 함수 및 가상 데이터셋 로더(k12_m01~m10) 준비 완료!")



---

## 🎯 미션 01. 자주 묻는 행정 질문 파악하기
**소요시간**: 약 1.5시간

![미션 1 자주 묻는 행정 질문 7대 영역 분류 체계 및 다빈도 TOP 5 우선순위 분석 다이어그램](/uploads/common/K12_m01_question_taxonomy.png)

### 📋 미션 개요 및 배경
휴가 신청, 증명서 발급 등 부대 행정 실무 현장에서 반복되는 구두 질의로 인한 행정 소요와 장병들의 정보 접근 애로사항을 분석합니다. 실습자료의 질문 카드 20개를 휴가·증명서·전입전출·자기개발·계정·시설 수리·기타 등 7대 영역으로 분류하고, 담당자가 가장 자주 직면하는 핵심 질문 5개를 도출합니다. 기존 구두 설명 및 게시판 응대 방식의 한계를 진단하여 생성형 AI 기반 행정 FAQ 챗봇이 해결해야 할 문제 정의와 도입 당위성을 명확히 수립합니다.

### 🎯 미션 학습목표
실습자료의 질문 카드 20개를 휴가·증명서·전입전출·자기개발·계정·시설 수리·기타로 분류하고, 담당자가 가장 자주 받을 것 같은 질문 5개를 고릅니다. 지금은 이런 질문에 어떻게 답하는지(담당자가 직접 설명, 게시판 등)와 불편한 점을 적습니다. **이를 통해 장병의 관점에서 행정 문제점을 정의하고, AI 챗봇 도입이 시급한 우선순위 요구사항을 분석·도출하는 현장 문제 정의 역량을 익힙니다.**

### 📑 실습 가이드 및 세부 단계 (프롬프트 & 검증 포인트)
#### 🔹 Step 1. 질문 카드 20개를 7개 분류로 나눈다.
#### 이 단계의 역할

가상 한빛대대 장병들이 제기하는 20개 행정 질문 카드를 휴가·증명서·전입전출·자기개발·계정·시설 수리·기타 7대 영역으로 체계적으로 분류합니다.

#### 해야 할 일

- 실습 데이터(JSON)에 수록된 20개 행정 질문 카드 원문을 꼼꼼히 검토합니다.

- 7대 행정 분류 기준에 맞춰 각 질문 카드를 배치하여 질문 분류표를 작성합니다.

- 복합적인 질문이나 경계가 모호한 질의는 AI에게 자문을 구하여 최적의 분류 항목을 확정합니다.

#### 💡 AI 프롬프트 질의 예시

> **[행정 질문 7대 영역 분류 질의]:** "너는 군 행정 업무 전문가야. 제공된 가상 장병 질문 카드 20건을 [휴가, 증명서, 전입전출, 자기개발, 계정, 시설 수리, 기타] 7개 영역으로 분류하고, 복합적인 질의는 주된 행정 목적에 따른 분류 근거를 함께 제시해 줘.
[질문 카드 목록]: {질문 카드 20개 원문}"

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **누락 없는 전수 분류:** 20개 질문 카드가 중복이나 누락 없이 7개 카테고리에 모두 배정되었는가?
- **카테고리 타당성:** 계정 초기화(전산)와 시설 수리(보급/영외), 증명서 발급과 휴가 신청이 명확히 구분되었는가?

#### 핵심 확인 지표

> 질문 카드 20개 7대 영역 분류 완료표

#### 학습자가 제출해야 하는 내용

- 질문 카드 20개 분류표 완성본

#### 🔹 Step 2. 담당자가 자주 받을 질문 5개를 고른다.
#### 이 단계의 역할

20개 질문 중 행정병 및 담당 실무자가 일과 중 가장 빈번하게 직면하는 다빈도 핵심 질문 5개를 선별합니다.

#### 해야 할 일

- 행정 문의 빈도와 계절/시점별 행정 소요(정기 휴가철, 자격증 접수 기간 등)를 종합 평가합니다.

- 질문 카드 20개 중 행정 응대 시간이 가장 많이 소요되는 5개 우선 질문을 선정합니다.

- AI에게 행정 실무자 관점에서의 다빈도 질의 우선순위 추천을 요청하고 본인 선별안과 교차 검증합니다.

#### 💡 AI 프롬프트 질의 예시

> **[행정 다빈도 우선 질문 선별 질의]:** "대대 행정병 입장에서 일과 시간 중 가장 자주 반복 문의를 받고 응대 소요가 큰 다빈도 핵심 행정 질문 5개를 추천해 줘. 각 질문별로 왜 반복 문의가 발생하기 쉬운지 이유를 분석해 줘.
[분류된 질문 목록]: {7대 분류 20개 질문}"

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **우선순위 현실성:** 휴가 신청 기한, 증명서 발급 등 실제 장병들의 수요가 높은 질의가 포함되었는가?
- **선정 타당성:** 선별된 5개 질문에 대한 명확한 업무 소요 및 빈도 근거가 기술되었는가?

#### 핵심 확인 지표

> 우선 질문 5선 및 선정 사유서

#### 학습자가 제출해야 하는 내용

- 우선 질문 5개 목록 및 선정 근거서

#### 🔹 Step 3. 지금의 응대 방식과 불편한 점을 적는다.
#### 이 단계의 역할

현재 행정반 방문 구두 문의, 전화 질의, 행정 게시판 등 기존 응대 방식의 비효율과 행정 공백 문제를 분석합니다.

#### 해야 할 일

- 장병의 행정반 직접 방문, 내선 전화 문의, 지휘계통 구두 질의 등 현재 응대 방식의 소요 시간을 정리합니다.

- 담당자 부재 시 확인 지연, 구두 전파로 인한 서류 누락 등 현장 문제점 3가지를 구체화합니다.

- 생성형 AI FAQ 챗봇 도입 시 기대되는 행정 개선 효과(업무 집중도 향상, 24시간 즉시 확인 등)를 도출합니다.

#### 💡 AI 프롬프트 질의 예시

> **[기존 행정 응대 한계 및 AI 도입 효과 질의]:** "군 행정 업무에서 장병들이 매번 행정반에 찾아와 구두로 묻거나 전화로 문의할 때 발생하는 담당자의 업무 피로와 장병들의 불편점을 3가지 도출해 줘. 이를 24시간 가동되는 행정 FAQ 챗봇으로 전환했을 때의 기대효과를 정리해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **실무 애로사항 반영:** 실제 부대 행정 현장에서 겪는 반복 질의 및 서류 보완 불편이 사실적으로 기술되었는가?
- **문제와 목적의 연계:** 도출된 문제점이 챗봇의 필요성과 목표 기능으로 자연스럽게 연결되는가?

#### 핵심 확인 지표

> 행정 응대 방식 비교 분석표 및 문제 진단서

#### 학습자가 제출해야 하는 내용

- 현재 행정 응대 방식과 불편점 정리서

#### 🔹 Step 4. 심화 도전 과제 및 최적화
#### 이 단계의 역할

기본 질문 20개 외에 실제 부대 행정에서 헷갈리기 쉬운 복합 행정 절차 질문 2건을 직접 고안하여 분류 체계를 확장합니다.

#### 해야 할 일

- 휴가와 위로휴가 연계, 복수 증명서 동시 발급 등 실제 자주 혼선을 겪는 심화 질문 2건을 구성합니다.

- AI에게 추가 질문의 적합성과 챗봇 반영 시 주의해야 할 절차적 쟁점을 자문합니다.

#### 💡 AI 프롬프트 질의 예시

> **[심화 행정 질문 개발 질의]:** "부대 행정 FAQ에 추가할 만한 헷갈리기 쉬운 질문 2건(예: 휴가 중 병원 진료 영수증 청구 절차, 자격증 응시료 사후 환급 절차)을 검토해 줘. 이 질문들이 장병 행정 편의에 미치는 영향과 FAQ에 포함할 핵심 항목을 조언해 줘."

#### 🔍 검증 포인트 (원문 대조 및 팩트체크)

- **보안 규정 준수:** 실제 부대 고유 명칭, 실제 인명, 비인가 규정 문서가 포함되지 않았는가?
- **실효성:** 추가된 질문이 장병들의 실제 행정 절차 이해에 기여하는가?

#### 핵심 확인 지표

> 심화 행정 질문 2건 추가 및 분석 메모

#### 학습자가 제출해야 하는 내용

- 심화 도전 과제 수행 결과 메모

### 📦 미션 필수 학습자료 및 팁
**미션 수행 필수 학습자료**

- **가상 한빛대대 행정 질문 카드 20선 (JSON/MD):** 휴가, 증명서, 전입전출, 자기개발, 계정, 시설수리 등 장병 다빈도 행정 질의 원문 데이터셋
- **행정 질문 7대 영역 분류 및 우선순위 분석표 서식 (MD/XLSX):** 7개 카테고리 분류 및 Top 5 다빈도 행정 질문 도출 시트
- **[학습 가이드] 행정 업무 프로세스 분석 및 페르소나 문제 정의서:** 행정병과 일반 장병의 관점에서 병목 현상을 진단하고 챗봇 도입 우선순위를 분석하는 실무 가이드

**실습 팁:** 20개 질문을 분류할 때는 장병들이 '어디에 신청하는가(부서)', '어떤 권리/서류인가(휴가/증명서)', '시설/전산 문제인가(계정/시설수리)'를 기준으로 나누면 명확합니다.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 문제 정의 — 장병 행정 질문 20개를 7대 영역으로 분류하고, 다빈도 우선 질문 5선과 AI 챗봇 도입 당위성을 도출했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 20개 행정 질문 카드 7대 영역 분류표 완성

- 다빈도 우선 질문 5선 및 선정 사유 타당성 확보

- 기존 응대 방식(방문/전화)의 한계 3가지 및 챗봇 도입 기대효과 정리

- 교육용 가상 데이터셋 활용 및 군사보안 수칙 엄수

#### 채점자 판정 메모

> 단순 질문 나열에 그치지 않고, 행정병의 반복 문의 피로와 장병들의 서류 미비 반려 문제를 정확히 진단하고 챗봇 구축 목적과 연결했는지 확인한다.



In [ ]:
# [미션 01.] 학생 실습 - 프롬프트 및 결과 기록 셀
# 💡 [데이터셋 확인 팁]: 아래 주석을 해제하여 본 미션의 가상 데이터셋 요약을 확인할 수 있습니다.
# show_dataset_summary(1)

# TODO 1: 위 실습 가이드와 AI 질의 예시를 참고하여 작성한 프롬프트를 입력하세요.
mission_01_prompt = """
# 여기에 본인이 설계한 프롬프트를 작성하세요.
"""

# TODO 2: 생성형 AI가 출력한 답변 및 원문 대조(Fact-check) 검증 내역을 기록하세요.
mission_01_result = """
# 여기에 AI 생성 결과 및 팩트체크 대조표/산출물을 마크다운 서식으로 기록하세요.
"""

# 렌더러 실행 (작성한 내용을 시각적으로 확인합니다)
show_result("미션 01. 자주 묻는 행정 질문 파악하기", mission_01_prompt, mission_01_result)

